In [2]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import StratifiedKFold,GridSearchCV
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
from sklearn.compose import ColumnTransformer,make_column_selector
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
import os
os.chdir("/home/pgcp-ai/MachineLearning/Datasets/")

In [3]:
hr = pd.read_csv("HR_comma_sep.csv")
hr

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low
...,...,...,...,...,...,...,...,...,...,...
14990,0.40,0.57,2,151,3,0,1,0,support,low
14991,0.37,0.48,2,160,3,0,1,0,support,low
14992,0.37,0.53,2,143,3,0,1,0,support,low
14993,0.11,0.96,6,280,4,0,1,0,support,low


In [4]:
X,y = hr.drop('left',axis=1),hr['left']

In [5]:
kfold = StratifiedKFold(n_splits=5,random_state=26,shuffle=True)

In [10]:
ohe = OneHotEncoder()
transformer = ColumnTransformer([('OHE',ohe,make_column_selector(dtype_include=object))],remainder='passthrough',verbose_feature_names_out=False)
rf = RandomForestClassifier(random_state=26)
pipe = Pipeline([('OHE',transformer),('RF',rf)])

In [13]:
params = {
    'RF__max_features' : np.arange(1,10),
    'RF__max_depth' : np.arange(1,7),
    'RF__n_estimators' : [10,25,100,150]
}
gcv = GridSearchCV(estimator=pipe,param_grid=params,cv=kfold)
gcv.fit(X,y)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=26, shuffle=True),
             estimator=Pipeline(steps=[('OHE',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('OHE',
                                                                         OneHotEncoder(),
                                                                         <sklearn.compose._column_transformer.make_column_selector object at 0x7f96cb0b34f0>)],
                                                          verbose_feature_names_out=False)),
                                       ('RF',
                                        RandomForestClassifier(random_state=26))]),
             param_grid={'RF__max_depth': array([1, 2, 3, 4, 5, 6]),
                         'RF__max_features': array([1, 2, 3, 4, 5, 6, 7, 8, 9]),
                         'RF__n_estimators': [10, 25, 100, 150]})

In [14]:
gcv.best_params_,gcv.best_score_

({'RF__max_depth': 6, 'RF__max_features': 9, 'RF__n_estimators': 150},
 0.9775258419473157)

In [15]:
bm = gcv.best_estimator_

In [5]:
import joblib

In [17]:
outfile = open('bm_rf.pkl','wb')
joblib.dump(bm,outfile)
outfile.close()

In [29]:
hr["number_project"].unique()

array([2, 5, 6, 4, 7, 3])

In [3]:
os.chdir("/home/pgcp-ai/MachineLearning/Models/")

In [6]:
infile = open("bm_rf.pkl", "rb")
model = joblib.load(infile)

In [7]:
os.chdir("/home/pgcp-ai/MachineLearning/Datasets/")

In [8]:
tst = pd.read_csv("tst_hr.csv")
model.predict(tst)

array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])